In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [1]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
agent_tools = Tools()
#agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(search)

In [7]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [11]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search funtion.
Use as many keywords from the user question as possible when making first request.

Make multimple searches.

Try to expand your search by using new key words based on the results you get from the search.

At the end ask  if there are othe areas that the user wants to explore.
"""

In [13]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [20]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [17]:
result.cost

CostInfo(input_cost=Decimal('0.00233025'), output_cost=Decimal('0.0008775'), total_cost=Decimal('0.00320775'))

In [21]:
result.all_messages


[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search funtion.\nUse as many keywords from the user question as possible when making first request.\n\nMake multimple searches.\n\nTry to expand your search by using new key words based on the results you get from the search.\n\nAt the end ask  if there are othe areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local install use ollama run model start service FAQ"}', call_id='call_OE7dp0R8obGwqkIuQ71lzJrD', name='search', type='function_call', id='fc_0781960d3ac5062e006a5ab5486b748199b6c02e1cecd78151', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_OE7dp0R8obGw

In [22]:
result2 = runner.loop(
    prompt="show me how to use Ollama with a Jupyter notebook",
    previous_messages=result.all_messages,
    callback=callback
)

-> Response received


-> Response received


In [23]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search funtion.\nUse as many keywords from the user question as possible when making first request.\n\nMake multimple searches.\n\nTry to expand your search by using new key words based on the results you get from the search.\n\nAt the end ask  if there are othe areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='how do I run olama?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"olama run ollama how do I run ollama install start server FAQ"}', call_id='call_LwGKifXiRO3SmkRhB5lYkfvo', name='search', type='function_call', id='fc_076218186e95ba27006a5ab687ce74819b9038d476a9ada524', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"how to run ollama com